数据集的格式：

单类别转换：

In [ ]:
import os
import json
import cv2
from tqdm import tqdm

def convert_json_to_yolo(input_dir, output_dir, category_name="all_fish"):
    """
    将 JSON 标注文件转换为 YOLO 格式的TXT文件（单类别专用）
    :param input_dir: 输入目录，包含JSON文件
    :param output_dir: YOLO格式输出文件夹
    :param category_name: JSON文件中使用的类别名称（默认"all_fish"）
    """
    # 创建输出目录
    os.makedirs(output_dir, exist_ok=True)

    # 固定类别ID为0（单类别）
    category_id = 0

    # 遍历输入目录中的所有JSON文件
    json_files = [f for f in os.listdir(input_dir) if f.endswith('.json')]
    
    for json_file in tqdm(json_files, desc="Processing JSON files"):
        json_path = os.path.join(input_dir, json_file)
        
        # 读取JSON文件
        with open(json_path, 'r') as f:
            try:
                data = json.load(f)
            except json.JSONDecodeError:
                print(f"Error: Invalid JSON format in {json_file}. Skipping...")
                continue

        # 获取图片尺寸信息
        if 'imageHeight' in data and 'imageWidth' in data:
            img_height = data['imageHeight']
            img_width = data['imageWidth']
        else:
            print(f"Warning: Missing image size info in {json_file}. Using default size 1920x1080")
            img_width, img_height = 1920, 1080

        # YOLO标注文件名
        txt_file = os.path.join(output_dir, f"{os.path.splitext(json_file)[0]}.txt")

        # 处理每个标注对象
        with open(txt_file, 'w') as f:
            if 'shapes' not in data:
                print(f"Warning: No 'shapes' field in {json_file}. Skipping...")
                continue
                
            for shape in data['shapes']:
                if 'label' not in shape or 'points' not in shape:
                    print(f"Warning: Invalid shape format in {json_file}. Skipping...")
                    continue
                    
                # 检查类别名称是否匹配（不区分大小写）
                if shape['label'].lower() != category_name.lower():
                    print(f"Warning: Unexpected category '{shape['label']}' in {json_file}. Using '{category_name}' instead.")
                
                points = shape['points']

                # 将多边形转为最小外接矩形
                x_coords = [p[0] for p in points]
                y_coords = [p[1] for p in points]
                x_min = min(x_coords)
                y_min = min(y_coords)
                x_max = max(x_coords)
                y_max = max(y_coords)

                # 计算中心点和宽高
                x_center = (x_min + x_max) / 2
                y_center = (y_min + y_max) / 2
                width = x_max - x_min
                height = y_max - y_min

                # 归一化坐标
                x_center /= img_width
                y_center /= img_height
                width /= img_width
                height /= img_height

                # 确保坐标在[0,1]范围内
                x_center = max(0, min(1, x_center))
                y_center = max(0, min(1, y_center))
                width = max(0, min(1, width))
                height = max(0, min(1, height))

                # 写入YOLO格式标注
                f.write(f"{category_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")

    print(f"转换完成！共处理 {len(json_files)} 个文件，输出保存在 {output_dir}")

# 示例用法
input_dir = "testDatasets/Json"  # JSON文件目录
output_dir = "testDatasets/labels"  # 输出目录

os.makedirs(output_dir, exist_ok=True)  # 确保输出目录存在

# 调用转换函数（单类别，默认使用"all_fish"作为类别名）
convert_json_to_yolo(input_dir, output_dir, category_name="all_fish")

多类别转换：(这个报错，之后再来看)

In [ ]:
import os
import json
import cv2
from tqdm import tqdm

def convert_labelme_to_yolo(input_dir, output_dir, category_to_id):
    """
    将 LabelMe 格式的 JSON 标注文件转换为 YOLO 格式（JSON 和图片在同一目录）
    :param input_dir: 输入目录，包含图片和 JSON 文件
    :param output_dir: YOLO 格式输出文件夹
    :param category_to_id: 类别到 ID 的映射
    """
    # 创建输出目录
    os.makedirs(output_dir, exist_ok=True)

    # 遍历输入目录
    for file in tqdm(os.listdir(input_dir), desc="Processing files"):
        # 跳过非 JSON 文件
        if not file.endswith('.json'):
            continue

        json_path = os.path.join(input_dir, file)
        with open(json_path, 'r') as f:
            data = json.load(f)

        # 获取对应图片路径
        image_name = data.get('imagePath', file.replace('.json', '.jpg'))  # 若 imagePath 缺失，假设图片是 .jpg
        image_name = os.path.splitext(image_name)[0] + '.jpg'  # 强制确保扩展名为 .jpg
        image_path = os.path.join(input_dir, image_name)

        if not os.path.exists(image_path):
            print(f"Warning: Image file {image_name} not found for {file}. Skipping...")
            continue

        # 读取图片尺寸
        img = cv2.imread(image_path)
        if img is None:
            print(f"Warning: Failed to load image {image_name}. Skipping...")
            continue
        img_height, img_width, _ = img.shape

        # YOLO 标注文件名
        label_file = os.path.join(output_dir, f"{os.path.splitext(file)[0]}.txt")

        # 处理每个标注对象
        with open(label_file, 'w') as f:
            for shape in data['shapes']:
                label = shape['label']
                points = shape['points']

                # 跳过未定义的类别
                if label not in category_to_id:
                    print(f"Warning: Undefined category '{label}' in {file}. Skipping...")
                    continue

                # 获取类别 ID
                category_id = category_to_id[label]

                # 将多边形转为最小外接矩形
                x_coords = [p[0] for p in points]
                y_coords = [p[1] for p in points]
                x_min = min(x_coords)
                y_min = min(y_coords)
                x_max = max(x_coords)
                y_max = max(y_coords)

                # 计算中心点和宽高
                x_center = (x_min + x_max) / 2
                y_center = (y_min + y_max) / 2
                width = x_max - x_min
                height = y_max - y_min

                # 归一化坐标
                x_center /= img_width
                y_center /= img_height
                width /= img_width
                height /= img_height

                # 写入 YOLO 格式标注
                f.write(f"{category_id} {x_center} {y_center} {width} {height}\n")

    print(f"标注转换完成！YOLO 格式标注保存在 {output_dir}")


# 示例用法
input_dir = "/App/cbw/Datasets/UnderWaterFish/Fish_hualu_ori/train/Labels"  # 文件夹路径，包含 JSON 和图片文件
output_dir = "dataset/hualu/labels/train"  # 输出的 YOLO 格式标注文件夹

# 类别到 ID 的映射（根据你的类别调整）
category_to_id = {
    "all_fish": 0  # 假设你只有一个类别 "all_fish"
}

convert_labelme_to_yolo(input_dir, output_dir, category_to_id)